# Sectorizar cooperativas con excel de ref.

In [1]:
import pandas as pd

# abrir excel con la clasificacion

In [2]:
clasificacion=pd.read_excel("../tablas/12_estados_financieros_e_indicadores_dic_2021_0.xlsx",skiprows=7)
clasificacion=clasificacion[["Entidad","Segmento"]]
clasificacion.tail(15)
clasisificacion_acortado= clasificacion.drop(clasificacion.tail(14).index)
clasisificacion_acortado

,Entidad,Segmento
0,COOPCAFAM,Medianas
1,COOPINDUMIL,Micro 1
2,COASMEDAS,Megas
3,BENEFICIAR,Grandes
4,COOPEBIS,Medianas
...,...,...
171,AFROAMERICANA,Micro 2
172,COOPCANAPRO,Pequeñas
173,SUCREDITO,Micro 1
174,CREDIAHORROS TAX FERIA*,Micro 1


In [3]:
#convertir dos columnas de una tabla a un diccionario
diccionario_clasificacion = dict(zip(clasisificacion_acortado["Entidad"], clasisificacion_acortado["Segmento"]))
diccionario_clasificacion

{'COOPCAFAM': 'Medianas',
 'COOPINDUMIL': 'Micro 1',
 'COASMEDAS': 'Megas',
 'BENEFICIAR': 'Grandes',
 'COOPEBIS': 'Medianas',
 'COOPSANFRANCISCO': 'Micro 2',
 'COOPEDAC': 'Medianas',
 'CODECOL': 'Micro 1',
 'PROGRESSA': 'Grandes',
 'COOPERATIVA AVP': 'Micro 2',
 'COOPFEBOR': 'Grandes',
 'COOPROFESORESUN': 'Medianas',
 'CREDICOOP': 'Medianas',
 'COOPEXXONMOBIL': 'Micro 2',
 'COOPSURAMERICA': 'Micro 1',
 'FINANCIAR': 'Micro 1',
 'COOTRAPELDAR': 'Medianas',
 'ALIANZA': 'Medianas',
 'CODEMA': 'Top 4',
 'CREDIFLORES': 'Grandes',
 'COOPCHIPAQUE': 'Micro 1',
 'USTACOOP LTDA.': 'Micro 1',
 'COOPETROL': 'Megas',
 'COOPETEXAS': 'Pequeñas',
 'COOPTRAISS': 'Megas',
 'BADIVENCOOP LTDA.': 'Pequeñas',
 'COOINDEGABO': 'Micro 1',
 'COPROCENVA': 'Megas',
 'ALCALICOOP': 'Micro 1',
 'COOVITEL': 'Medianas',
 'COOPTENJO': 'Grandes',
 'COOACUEDUCTO': 'Medianas',
 'COOLEVER': 'Micro 1',
 'CIDESA': 'Micro 1',
 'COOPEREN': 'Micro 1',
 'COTRAMED': 'Micro 1',
 'COOBELMIRA': 'Micro 2',
 'CODELCO': 'Micro 2',
 'CO

# abrir codigo 

In [4]:
# abrimos el excel con los datos de las cooperativas y lo guardamos en un dataframe
df_coperativas = pd.read_excel("../tablas/registros.xlsx")
df_coperativas

,ID_registro,ID_indicador,ID_cooperativa,ano,mes,valor
0,1,Quebranto Patrimonial,CAJA COOPERATIVA CREDICOOP,2014,1,1.500449
1,2,Quebranto Patrimonial,CAJA COOPERATIVA PETROLERA,2014,1,1.310736
2,3,Quebranto Patrimonial,CASA NACIONAL DEL PROFESOR,2014,1,1.188087
3,4,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,2014,1,1.588034
4,5,Quebranto Patrimonial,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,2014,1,1.481415
...,...,...,...,...,...,...
213305,213306,Indicador de Riesgo de Liquidez - IRL,COPROCENVA COOPERATIVA DE AHORRO Y CREDITO,2025,12,0.000000
213306,213307,Indicador de Riesgo de Liquidez - IRL,EMPRESA COOPERATIVA DE AHORRO Y CREDITO SIGLO ...,2025,12,0.000000
213307,213308,Indicador de Riesgo de Liquidez - IRL,FINANCIERA COOPERATIVA COLOMBIANA DE INGENIEROS,2025,12,0.000000
213308,213309,Indicador de Riesgo de Liquidez - IRL,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURS...,2025,12,0.000000


# Función de matching entre diccionario y lista

Función que hace matching flexible entre las keys de un diccionario y los valores de una lista

In [5]:
# Importaciones necesarias
import re
from unidecode import unidecode
from difflib import SequenceMatcher

In [6]:
def normalizar_texto(texto):
    """
    Normaliza un texto para facilitar comparaciones.
    
    Args:
        texto (str): Texto a normalizar
    
    Returns:
        str: Texto normalizado (minúsculas, sin tildes, sin caracteres especiales)
    """
    if pd.isna(texto) or texto is None:
        return ""
    
    # Convertir a string y minúsculas
    texto = str(texto).lower().strip()
    
    # Eliminar tildes
    texto = unidecode(texto)
    
    # Eliminar palabras comunes
    palabras_comunes = [
        r'\bcooperativa\b', r'\bcoop\b', r'\bcoop\.\b',
        r'\bltda\b', r'\bltda\.\b', r'\blimitada\b',
        r'\bde\b', r'\bdel\b', r'\bla\b', r'\bel\b',
        r'\blos\b', r'\blas\b', r'\by\b', r'\ba\b'
    ]
    
    for palabra in palabras_comunes:
        texto = re.sub(palabra, ' ', texto)
    
    # Eliminar caracteres especiales (solo letras, números y espacios)
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    
    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

In [7]:
def calcular_similitud(texto1, texto2):
    """
    Calcula la similitud entre dos textos usando SequenceMatcher.
    
    Args:
        texto1 (str): Primer texto
        texto2 (str): Segundo texto
    
    Returns:
        float: Similitud entre 0 y 1
    """
    return SequenceMatcher(None, texto1, texto2).ratio()

In [8]:
def matchear_diccionario_con_lista(diccionario, lista_valores, umbral_similitud=0.7, mostrar_detalles=False):
    """
    Hace matching flexible entre las keys de un diccionario y los valores de una lista.
    
    Estrategias de matching:
    1. Match exacto de textos normalizados
    2. Match parcial (contiene o está contenido)
    3. Match por similitud de secuencia (usando SequenceMatcher)
    4. Match por palabras clave comunes (similitud Jaccard)
    
    Args:
        diccionario (dict): Diccionario original {key: value}
        lista_valores (list): Lista de valores a buscar en las keys
        umbral_similitud (float): Umbral mínimo de similitud (0-1). Default: 0.7
        mostrar_detalles (bool): Si True, muestra los detalles del matching
    
    Returns:
        dict: Nuevo diccionario con keys de la lista que matchearon y valores originales
              Formato: {valor_de_lista: valor_original_diccionario}
    
    Ejemplo:
        >>> dict_original = {"COOPERATIVA DE MEDELLÍN LTDA": "Segmento A", 
                             "COOP BOGOTÁ": "Segmento B"}
        >>> lista = ["Cooperativa Medellin", "Coop de Bogota"]
        >>> resultado = matchear_diccionario_con_lista(dict_original, lista)
        >>> # Resultado: {"Cooperativa Medellin": "Segmento A", "Coop de Bogota": "Segmento B"}
    """
    
    # Validaciones
    if not isinstance(diccionario, dict):
        raise TypeError("El primer argumento debe ser un diccionario")
    if not isinstance(lista_valores, (list, pd.Series)):
        raise TypeError("El segundo argumento debe ser una lista o Series de pandas")
    
    # Convertir Series a lista si es necesario
    if isinstance(lista_valores, pd.Series):
        lista_valores = lista_valores.tolist()
    
    # Diccionario de resultados
    resultado = {}
    
    # Diccionario con keys normalizadas para búsqueda rápida
    dict_normalizado = {
        normalizar_texto(k): (k, v) 
        for k, v in diccionario.items()
    }
    
    # Detalles del matching (opcional)
    detalles = []
    
    # Procesar cada valor de la lista
    for valor_lista in lista_valores:
        if pd.isna(valor_lista) or valor_lista is None or valor_lista == "":
            continue
        
        valor_norm = normalizar_texto(valor_lista)
        
        if not valor_norm:
            continue
        
        match_encontrado = False
        tipo_match = "sin_match"
        key_original = None
        valor_dict = None
        score = 0.0
        
        # Estrategia 1: Match exacto
        if valor_norm in dict_normalizado:
            key_original, valor_dict = dict_normalizado[valor_norm]
            tipo_match = "exacto"
            score = 1.0
            match_encontrado = True
        
        # Estrategia 2: Match parcial (contiene o está contenido)
        if not match_encontrado:
            for key_dict_norm, (key_orig, val_orig) in dict_normalizado.items():
                if not key_dict_norm:
                    continue
                
                # Si el valor de la lista contiene la key del diccionario
                if key_dict_norm in valor_norm:
                    key_original = key_orig
                    valor_dict = val_orig
                    tipo_match = "contiene"
                    score = len(key_dict_norm) / len(valor_norm)
                    match_encontrado = True
                    break
                
                # Si la key del diccionario contiene el valor de la lista
                if valor_norm in key_dict_norm:
                    key_original = key_orig
                    valor_dict = val_orig
                    tipo_match = "contenido_en"
                    score = len(valor_norm) / len(key_dict_norm)
                    match_encontrado = True
                    break
        
        # Estrategia 3: Match por similitud de secuencia
        if not match_encontrado:
            mejor_score_seq = 0
            mejor_match_seq = None
            
            for key_dict_norm, (key_orig, val_orig) in dict_normalizado.items():
                if not key_dict_norm:
                    continue
                
                similitud_seq = calcular_similitud(valor_norm, key_dict_norm)
                
                if similitud_seq > mejor_score_seq and similitud_seq >= umbral_similitud:
                    mejor_score_seq = similitud_seq
                    mejor_match_seq = (key_orig, val_orig)
            
            if mejor_match_seq:
                key_original, valor_dict = mejor_match_seq
                tipo_match = "similitud_secuencia"
                score = mejor_score_seq
                match_encontrado = True
        
        # Estrategia 4: Match por palabras clave (Jaccard)
        if not match_encontrado:
            palabras_valor = set(valor_norm.split())
            if not palabras_valor:
                continue
            
            mejor_score_jaccard = 0
            mejor_match_jaccard = None
            
            for key_dict_norm, (key_orig, val_orig) in dict_normalizado.items():
                palabras_key = set(key_dict_norm.split())
                if not palabras_key:
                    continue
                
                interseccion = palabras_valor.intersection(palabras_key)
                union = palabras_valor.union(palabras_key)
                
                if union:
                    similitud_jaccard = len(interseccion) / len(union)
                    
                    if similitud_jaccard > mejor_score_jaccard and similitud_jaccard >= umbral_similitud:
                        mejor_score_jaccard = similitud_jaccard
                        mejor_match_jaccard = (key_orig, val_orig)
            
            if mejor_match_jaccard:
                key_original, valor_dict = mejor_match_jaccard
                tipo_match = "similitud_jaccard"
                score = mejor_score_jaccard
                match_encontrado = True
        
        # Guardar resultado si se encontró match
        if match_encontrado and valor_dict is not None:
            resultado[valor_lista] = valor_dict
            detalles.append({
                'valor_lista': valor_lista,
                'key_diccionario': key_original,
                'valor_asignado': valor_dict,
                'tipo_match': tipo_match,
                'score': score
            })
        else:
            detalles.append({
                'valor_lista': valor_lista,
                'key_diccionario': None,
                'valor_asignado': None,
                'tipo_match': 'sin_match',
                'score': 0.0
            })
    
    # Mostrar detalles si se solicita
    if mostrar_detalles:
        print("\n" + "="*80)
        print(f"DETALLES DEL MATCHING")
        print("="*80)
        print(f"\nTotal valores en lista: {len(lista_valores)}")
        print(f"Total keys en diccionario: {len(diccionario)}")
        print(f"Matches encontrados: {len(resultado)}")
        print(f"Sin match: {len([d for d in detalles if d['tipo_match'] == 'sin_match'])}")
        
        print(f"\n{'Valor Lista':<30} | {'Key Dict':<30} | {'Tipo':<20} | {'Score':<6}")
        print("-"*80)
        
        for detalle in detalles[:20]:  # Mostrar primeros 20
            valor = detalle['valor_lista'][:28] if len(detalle['valor_lista']) > 28 else detalle['valor_lista']
            key = (detalle['key_diccionario'][:28] if detalle['key_diccionario'] and len(detalle['key_diccionario']) > 28 
                   else detalle['key_diccionario']) or "N/A"
            tipo = detalle['tipo_match']
            score = f"{detalle['score']:.2f}"
            print(f"{valor:<30} | {key:<30} | {tipo:<20} | {score:<6}")
        
        if len(detalles) > 20:
            print(f"\n... y {len(detalles) - 20} más")
    
    return resultado

## Ejemplo de uso

Veamos cómo usar la función con los datos que ya tienes:

In [12]:
# Ejemplo: Crear un nuevo diccionario con las cooperativas de df_coperativas 
# que matchean con las del diccionario_clasificacion

# Obtener la lista de cooperativas del DataFrame
lista_cooperativas = df_coperativas['ID_cooperativa'].unique().tolist()

# Aplicar el matching
diccionario_resultado = matchear_diccionario_con_lista(
    diccionario=diccionario_clasificacion,
    lista_valores=lista_cooperativas,
    umbral_similitud=0.3,
    mostrar_detalles=True
)

print("\n" + "="*80)
print("DICCIONARIO RESULTANTE")
print("="*80)
print(f"\nEntradas en diccionario resultado: {len(diccionario_resultado)}")
print("\nPrimeras 10 entradas:")
for i, (key, value) in enumerate(list(diccionario_resultado.items())[:10]):
    print(f"  {i+1}. '{key}' → '{value}'")


DETALLES DEL MATCHING

Total valores en lista: 88
Total keys en diccionario: 176
Matches encontrados: 88
Sin match: 0

Valor Lista                    | Key Dict                       | Tipo                 | Score 
--------------------------------------------------------------------------------
CAJA COOPERATIVA CREDICOOP     | CREDICOOP                      | contiene             | 0.64  
CAJA COOPERATIVA PETROLERA     | COOPETROL                      | similitud_secuencia  | 0.61  
CASA NACIONAL DEL PROFESOR     | COOPROFESORES                  | similitud_secuencia  | 0.57  
COOPANTEX COOPERATIVA DE AHO   | COOPANTEX                      | contiene             | 0.38  
COOPERATIVA AHORRO Y CREDITO   | COOPERATIVA DE AHORRO Y CRED   | similitud_secuencia  | 0.71  
COOPERATIVA BELEN AHORRO Y C   | COOPERATIVA DE AHORRO Y CRED   | similitud_secuencia  | 0.67  
COOPERATIVA CALDENSE DEL PRO   | COOPROFESORES                  | similitud_secuencia  | 0.60  
COOPERATIVA DE AHORRO Y CRED   

In [13]:
diccionario_resultado

{'CAJA COOPERATIVA CREDICOOP': 'Medianas',
 'CAJA COOPERATIVA PETROLERA': 'Megas',
 'CASA NACIONAL DEL PROFESOR': 'Megas',
 'COOPANTEX COOPERATIVA DE AHORRO Y CREDITO': 'Grandes',
 'COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.': 'Medianas',
 'COOPERATIVA BELEN AHORRO Y CREDITO': 'Medianas',
 'COOPERATIVA CALDENSE DEL PROFESOR': 'Megas',
 'COOPERATIVA DE AHORRO Y CREDITO BERLIN': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO COLANTA': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO CONGENTE': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO COOMPARTIR': 'Micro 2',
 'COOPERATIVA DE AHORRO Y CREDITO COOTRAIPI': 'Pequeñas',
 'COOPERATIVA DE AHORRO Y CREDITO COOYAMOR': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO CREAFAM': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO CREAR LTDA CREARCOP': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO DE AIPE': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO DE CHIPAQUE': 'Medianas',
 'COOPERATIVA DE AHORRO Y CREDITO DE DROGUISTAS DETALLISTAS': 'Medianas',
 'CO

## Aplicar el diccionario resultado al DataFrame

Ahora vamos a agregar la columna "categoria" al DataFrame df_coperativas usando el diccionario_resultado:

In [17]:
# Aplicar el diccionario_resultado para crear la columna 'categoria'
# Usamos .map() para mapear los valores de ID_cooperativa con el diccionario
df_coperativas['categoria'] = df_coperativas['ID_cooperativa'].map(diccionario_resultado)

# Llenar valores sin match con "Sin clasificar"
df_coperativas['categoria'] = df_coperativas['categoria'].fillna('Sin clasificar')

# Mostrar las primeras filas con la nueva columna
print("DataFrame actualizado con columna 'categoria':")
print("="*80)
df_coperativas[['ID_cooperativa', 'categoria']].head(15)

DataFrame actualizado con columna 'categoria':


,ID_cooperativa,categoria
0,CAJA COOPERATIVA CREDICOOP,Medianas
1,CAJA COOPERATIVA PETROLERA,Megas
2,CASA NACIONAL DEL PROFESOR,Megas
3,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,Grandes
4,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,Medianas
5,COOPERATIVA BELEN AHORRO Y CREDITO,Medianas
6,COOPERATIVA CALDENSE DEL PROFESOR,Megas
7,COOPERATIVA DE AHORRO Y CREDITO BERLIN,Medianas
8,COOPERATIVA DE AHORRO Y CREDITO COLANTA,Medianas
9,COOPERATIVA DE AHORRO Y CREDITO CONGENTE,Medianas


## Estadísticas de la categorización

In [19]:
# Ver estadísticas de la categorización
print("\nESTADÍSTICAS DE CATEGORIZACIÓN")
print("="*80)
print(f"\nTotal de registros: {len(df_coperativas)}")
print(f"Total de cooperativas únicas: {df_coperativas['ID_cooperativa'].nunique()}")
print(f"\nDistribución por categoría:")
print(df_coperativas['categoria'].value_counts())

# Calcular porcentajes
print(f"\n\nPorcentaje de registros categorizados:")
categorizados = df_coperativas[df_coperativas['categoria'] != 'Sin clasificar'].shape[0]
porcentaje = (categorizados / len(df_coperativas)) * 100
print(f"  Categorizados: {categorizados} ({porcentaje:.2f}%)")
print(f"  Sin clasificar: {len(df_coperativas) - categorizados} ({100-porcentaje:.2f}%)")


ESTADÍSTICAS DE CATEGORIZACIÓN

Total de registros: 213310
Total de cooperativas únicas: 88

Distribución por categoría:
categoria
Medianas    80050
Micro 1     41216
Micro 2     33880
Grandes     24278
Pequeñas    19368
Megas       12092
Top 4        2426
Name: count, dtype: int64


Porcentaje de registros categorizados:
  Categorizados: 213310 (100.00%)
  Sin clasificar: 0 (0.00%)


## Ver registros sin categorizar (opcional)

Si hay cooperativas sin categoría, las mostramos:

In [20]:
# Mostrar cooperativas sin clasificar
sin_categoria = df_coperativas[df_coperativas['categoria'] == 'Sin clasificar']['ID_cooperativa'].unique()

print(f"\nCOOPERATIVAS SIN CATEGORÍA: {len(sin_categoria)}")
print("="*80)

if len(sin_categoria) > 0:
    print("\nLista de cooperativas sin categoría:")
    for i, coop in enumerate(sin_categoria[:20], 1):
        print(f"  {i}. {coop}")
    
    if len(sin_categoria) > 20:
        print(f"\n... y {len(sin_categoria) - 20} más")
else:
    print("\n✅ ¡Excelente! Todas las cooperativas fueron categorizadas.")


COOPERATIVAS SIN CATEGORÍA: 0

✅ ¡Excelente! Todas las cooperativas fueron categorizadas.


## Ver el DataFrame completo

Visualizar el DataFrame actualizado:

In [30]:
# Mostrar el DataFrame completo con la nueva columna
df_coperativas.to_excel("../tablas/registros_categorizados.xlsx", index=False)
print("\nDataFrame completo con la nueva columna 'categoria' ha sido guardado en '../tablas/registros_categorizados.xlsx'")


DataFrame completo con la nueva columna 'categoria' ha sido guardado en '../tablas/registros_categorizados.xlsx'
